# Lab 3 — Image, Kernel, and Gaussian Elimination
**Linear Algebra · UATX**

For $A \in \mathbb{R}^{m \times n}$, column-reducing the augmented matrix $[A \mid I_n]$ simultaneously yields:
- A basis for $\mathrm{im}(A)$ (pivot columns of $A$)
- A basis for $\ker(A)$ (lower part of non-pivot columns)

Extending to $[A \mid I_n \mid {-b}]$ solves $Ax = b$ in the same pass.

**Tasks**
1. Run `column_reduce_image_kernel` on matrices with trivial and non-trivial kernels.
2. Use `gauss_solve` for all three cases: unique / infinitely many / no solution.
3. Explore the **Hilbert matrix**: invertible in exact arithmetic, numerically rank-deficient at $n=20$.
4. *(New)* Plot the numerical rank of $H_n$ for $n=2,4,\ldots,20$ and locate where floating-point fails.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sympy import Matrix, binomial, Rational

## §1  Column Reduction: $[A \mid I_n]$

Column operations on $[A \mid I_n]$ preserve $\mathrm{im}(A)$. After reduction:
- Pivot columns of $A$ form a basis for $\mathrm{im}(A)$.
- The $I_n$ block transforms: its columns at free positions give a basis for $\ker(A)$.

In [ ]:
def column_reduce_image_kernel(A, tol=1e-10):
    """
    Column-reduce [A ; I_n] to obtain bases for im(A) and ker(A).
    Returns (reduced, pivot_cols).
    reduced[:m] : echelon form; reduced[m:] : kernel in non-pivot cols.
    """
    m, n = A.shape
    AI = np.vstack([A.astype(float), np.eye(n)])
    used_rows, pivot_positions = set(), []
    for col in range(n):
        pivot_row = next((r for r in range(m) if r not in used_rows and abs(AI[r,col]) > tol), None)
        if pivot_row is None: continue
        used_rows.add(pivot_row); pivot_positions.append((pivot_row, col))
        AI[:, col] /= AI[pivot_row, col]
        for other in range(n):
            if other != col:
                AI[:, other] -= AI[pivot_row, other] * AI[:, col]
    pivot_positions.sort()
    pivot_cols = [c for r,c in pivot_positions]
    free_cols  = [j for j in range(n) if j not in pivot_cols]
    return AI[:, pivot_cols + free_cols], pivot_cols

In [ ]:
# Example 1: rank-2 matrix
A = np.array([[1., 2., 3.],
              [2., 4., 6.],
              [1., 1., 1.]], dtype=float)

reduced, pivots = column_reduce_image_kernel(A)
print('A (rank', np.linalg.matrix_rank(A), ')\nReduced form:')
print(np.round(reduced, 4))
print('Pivot cols (im basis):', pivots)

m = A.shape[0]
n_free = A.shape[1] - len(pivots)
if n_free > 0:
    ker_basis = reduced[m:, len(pivots):]
    print('ker(A) basis:', ker_basis.T)
    print('A @ ker_basis = 0?', np.allclose(A @ ker_basis, 0))

In [ ]:
# Example 2: invertible 3x3
A2 = np.array([[1.,2.,3.],[2.,4.,0.],[1.,1.,1.]], dtype=float)
reduced2, pivots2 = column_reduce_image_kernel(A2)
print('Invertible matrix, pivot cols:', pivots2)
A_inv = reduced2[3:, :3]   # bottom block = A^{-1}
print('A^{-1} (bottom block):\n', np.round(A_inv, 4))
print('A @ A^{-1} = I?', np.allclose(A2 @ A_inv, np.eye(3)))

## §2  Gaussian Elimination: solving $Ax = b$

In [ ]:
def gauss_solve(A, b, tol=1e-10):
    """
    Solve Ax=b by column reduction on [A ; I_n | -b ; 0].
    Returns (reduced, pivot_cols).
    """
    m, n = A.shape
    AI  = np.vstack([A.astype(float), np.eye(n)])
    rhs = np.hstack([b.astype(float), np.zeros(n)]).reshape(-1, 1)
    AIb = np.hstack([AI, -rhs])
    used_rows, pivot_positions = set(), []
    for col in range(n):
        for r in range(m):
            if r not in used_rows and abs(AIb[r, col]) > tol:
                used_rows.add(r); pivot_positions.append((r, col))
                AIb[:, col] /= AIb[r, col]
                for other in range(n):
                    if other != col:
                        AIb[:, other] -= AIb[r, other] * AIb[:, col]
                AIb[:, -1] -= AIb[r, -1] * AIb[:, col]
                break
    pivot_positions.sort()
    pivot_cols = [c for r,c in pivot_positions]
    free_cols  = [j for j in range(n) if j not in pivot_cols]
    return AIb[:, pivot_cols + free_cols + [n]], pivot_cols

In [ ]:
# Case A: invertible — unique solution
A_sq = np.array([[1.,2.,3.],[2.,4.,0.],[1.,1.,1.]], dtype=float)
b_ok = np.array([1.,2.,3.])
red, piv = gauss_solve(A_sq, b_ok)
print('=== Unique solution ===')
print('Reduced [A|I|-b]:\n', np.round(red, 4))
x_sol = red[3:len(piv)+3, len(piv)]   # RHS column of bottom block, pivot part
print('x =', np.round(red[3:, len(piv)], 4)[:len(piv)])
print('A x = b?', np.allclose(A_sq @ red[3:, len(piv)][:len(piv)], b_ok))

In [ ]:
# Case B: underdetermined consistent — infinitely many
A3 = np.array([[1.,2.,3.,4.],[2.,4.,6.,8.],[1.,1.,1.,1.]], dtype=float)
b3 = np.array([1.,2.,3.])
red3, piv3 = gauss_solve(A3, b3)
print('=== Infinitely many solutions ===')
print('rank =', len(piv3), ', dim ker =', 4 - len(piv3))
print('Reduced form:\n', np.round(red3, 4))

# Extract particular solution x0 and kernel vectors
x0  = red3[3:, len(piv3)][:4]   # last column, bottom n rows, pivot-reordered
ker = red3[3:, len(piv3)+1:-1]  # free columns (kernel)
print('x0 =', np.round(x0, 4), '  A x0 = b?', np.allclose(A3 @ x0, b3))
print('ker basis cols: A@ker = 0?', np.allclose(A3 @ ker, 0))

In [ ]:
# Case C: inconsistent — no solution
b_bad = np.array([1.,3.,1.])
red_bad, _ = gauss_solve(A3, b_bad)
print('=== No solution ===')
print('Reduced form:\n', np.round(red_bad, 4))
print('Non-zero residual in RHS (rows 0..m-1):', not np.allclose(red_bad[:3, -1], 0))

## §3  The Hilbert Matrix

$H_{ij} = 1/(i+j-1)$, $1$-indexed. Theoretically invertible for all $n$ (exact closed-form inverse exists). Yet floating-point arithmetic cannot detect its full rank for $n \geq 13$.

In [ ]:
def hilbert_matrix(n):
    return Matrix(n, n, lambda i, j: Rational(1, i + j + 1))

def hilbert_inverse(n):
    H_inv = Matrix.zeros(n)
    for i in range(n):
        for j in range(n):
            sign = (-1)**(i+j)
            a, b_ = i+1, j+1
            H_inv[i,j] = sign*(a+b_-1)*binomial(n+a-1,n-b_)*binomial(n+b_-1,n-a)*binomial(a+b_-2,a-1)**2
    return H_inv

# Verify for n=5
H5 = hilbert_matrix(5); H5_inv = hilbert_inverse(5)
print('n=5: H * H_inv = I?', H5 @ H5_inv == Matrix.eye(5))

# n=20: exact vs float
H20 = hilbert_matrix(20)
H20_inv = hilbert_inverse(20)
print('n=20 exact: H*H_inv=I?', H20@H20_inv == Matrix.eye(20))

H20_fp = np.array(H20.tolist(), dtype=np.float64)
print('n=20 float: np.matrix_rank =', np.linalg.matrix_rank(H20_fp), '(true rank = 20)')
_, pivs = column_reduce_image_kernel(H20_fp)
print('n=20 float: column_reduce rank =', len(pivs))

## §4  Rank Collapse of $H_n$ as $n$ Grows  *(New)*

At what size does the floating-point rank first drop below the true rank?

In [ ]:
sizes = list(range(2, 22, 2))
np_ranks, cond_numbers = [], []
for n in sizes:
    H_fp = np.array(hilbert_matrix(n).tolist(), dtype=np.float64)
    np_ranks.append(np.linalg.matrix_rank(H_fp))
    cond_numbers.append(np.linalg.cond(H_fp))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(sizes, sizes, 'g-o', lw=2, label='True rank $n$')
axes[0].plot(sizes, np_ranks, 'r--s', lw=2, label='np.matrix_rank (float64)')
axes[0].set_xlabel('$n$'); axes[0].set_ylabel('Rank')
axes[0].set_title('Hilbert matrix: true vs. numerical rank')
axes[0].legend(); axes[0].grid(True)

axes[1].semilogy(sizes, cond_numbers, 'k-o', lw=2)
axes[1].axhline(1/np.finfo(float).eps, color='red', ls='--',
                label=f'$1/\\epsilon_{{\\rm mach}} \\approx 10^{{16}}$')
axes[1].set_xlabel('$n$'); axes[1].set_ylabel('$\\kappa(H_n)$')
axes[1].set_title('Condition number of $H_n$')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout(); plt.show()

failure = next((n for n,r in zip(sizes, np_ranks) if r < n), None)
print(f'Rank first drops below n at n = {failure}')